In [ ]:
# Imports the 'os' module, which allows interaction with the operating system 
import os 

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' 
my_bucket


# COMMAND 

In [ ]:
%%writefile high_confidence_analysis.sh
#!/bin/bash

# Exit immediately if a command exits with a non-zero status
set -o errexit

# Treat unset variables as an error (prevents using undefined variables)
set -o nounset

# Loop through the populations
for pop in EUR; do
    echo "Processing population: ${pop}"
    
    # Loop through the phenotypes 
    for pheno in aml_biosample_age aml_diagnosis_age; do 
    
    echo "Processing pop: ${pop} | phenotype: ${pheno} | chr: ${chrom}"
    
    step2_SPAtests.R \
        --bedFile="${plink_bed}" \
        --bimFile="${plink_bim}" \
        --famFile="${plink_fam}" \
        --SAIGEOutputFile="${output}/${pop}_${pheno}_high_confidence_chr${chrom}" \
        --LOCO=FALSE \
        --AlleleOrder=alt-first \
        --minMAF=0 \
        --minMAC=0.5 \
        --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
        --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --groupFile="${vat}/${pop}_HC_annotation/HC-annotation-chr${chrom}.txt" \
        --annotation_in_groupTest=AlphaMissense-0.564,LOFTEE-HC:SpliceAI-0.8,LOFTEE-HC:SpliceAI-0.8:AlphaMissense-0.564 \
        --maxMAF_in_groupTest=0.0001,0.001,0.01 \
        --is_output_markerList_in_groupTest=TRUE \
        --is_fastTest=TRUE
                
    step2_SPAtests.R \
        --bedFile="${plink_bed}" \
        --bimFile="${plink_bim}" \
        --famFile="${plink_fam}" \
        --SAIGEOutputFile="${output}/${pop}_${pheno}_synonymous_chr${chrom}" \
        --LOCO=FALSE \
        --AlleleOrder=alt-first \
        --minMAF=0 \
        --minMAC=0.5 \
        --GMMATmodelFile="${null_glmm}/${pop}_${pheno}_null_glmm.rda" \
        --varianceRatioFile="${null_glmm}/${pop}_${pheno}_null_glmm.varianceRatio.txt" \
        --sparseGRMFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx" \
        --sparseGRMSampleIDFile="${grm}/${pop}_GRM_relatednessCutoff_0.05_5000_randomMarkersUsed.sparseGRM.mtx.sampleIDs.txt" \
        --groupFile="${vat}/${pop}_HC_annotation/HC-annotation-chr${chrom}.txt" \
        --annotation_in_groupTest=Synonymous \
        --maxMAF_in_groupTest=0.0001,0.001,0.01 \
        --is_output_markerList_in_groupTest=TRUE \
        --is_fastTest=TRUE
                
    done
done


In [ ]:
# Copy script in workspace bucket 
!gsutil cp high_confidence_analysis.sh {my_bucket}/data/scripts/

In [ ]:
# Check script in workspace bucket 
!gsutil cat {my_bucket}/data/scripts/high_confidence_analysis.sh

# SUBMISSION 

In [ ]:
import subprocess
import pandas as pd

chromosome = []
sizes = [] 

# Loop for all chromosome 
for chrom in range(1, 23):
    
    # Extract size of genomic plink bed file in GB for each chromosome 
    command = [
        "gsutil",
        "-u", os.getenv("GOOGLE_PROJECT"),
        "ls",
        "-lha",
        f"gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{chrom}.bed"
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    
    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size 
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

print(genomic_size)

In [ ]:

# Define environment variables 
WORKSPACE_BUCKET = os.getenv("WORKSPACE_BUCKET")
ARTIFACT_REGISTRY_DOCKER_REPO = os.getenv("ARTIFACT_REGISTRY_DOCKER_REPO")

# Define path to logging directory
LOGGING=f'{WORKSPACE_BUCKET}/data/logging'

# Define path to sparse genetic relationship matrix 
GRM=f'{WORKSPACE_BUCKET}/data/saige_step00/grm/'

# Define path to null model
NULL_GLMM=f'{WORKSPACE_BUCKET}/data/saige_step01/null_glmm/'

# Define path to annotation table 
VAT=f'{WORKSPACE_BUCKET}/data/annotation_table/'

# Define path to output directory 
OUTPUT=f'{WORKSPACE_BUCKET}/data/saige_step02/HC_analysis/'

# Define path to script 
SCRIPT=f'{WORKSPACE_BUCKET}/data/scripts/high_confidence_analysis.sh'

# Define path to mount path
MNT_PATH='/mnt/data/input/gs/fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed'

MAX_DISK_SIZE = 500 # Set maximum disk size
EXTRA_DISK_SIZE = 30 # Set additional disk size 

# Loop through each chromosome 
for CHROM in range(1,23):
    
    print(f"Submitting dsub command for chromosome {CHROM}")
    
    DISK_SIZE = round(float(genomic_size.loc[genomic_size['chromosome'] == CHROM, 'size'].iloc[0])) + EXTRA_DISK_SIZE
    
    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE: # Set maximum disk size 
        print(f"Disk too large for chromosome {CHROM}")
        break
        
    if DISK_SIZE > 1000: # Absolute maximum for version 8  
        print(f"Disk too large for chromosome {CHROM}")
        break

    BED=f'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{CHROM}.bed'
    BIM=f'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{CHROM}.bim'
    FAM=f'gs://fc-aou-datasets-controlled/v8/wgs/short_read/snpindel/exome/plink_bed/exome.chr{CHROM}.fam'

    command = [
        "aou_dsub",
        "--image", f"{ARTIFACT_REGISTRY_DOCKER_REPO}/wzhou88/saige:1.5.0.2",
        "--boot-disk-size", "10",
        "--disk-size", str(DISK_SIZE),
        "--min-ram", "4",
        "--logging", LOGGING,
        "--input", f"plink_bed={BED}",
        "--input", f"plink_bim={BIM}",
        "--input", f"plink_fam={FAM}",
        "--input-recursive", f"grm={GRM}",
        "--input-recursive", f"null_glmm={NULL_GLMM}",
        "--input-recursive", f"vat={VAT}",
        "--output-recursive", f"output={OUTPUT}",
        "--env", f"genos={MNT_PATH}",
        "--env", f"chrom={CHROM}",
        "--script", SCRIPT
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {' '.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)